## Route acquisition with the Routing API

The [previous three notebooks](obm_6_route.ipynb) build detour reranking out of the demo
classes (`RouteEngine`, `DetourRanker`, `ResponseMap`). This one and the next two
build the same feature out of a **single low-level class**, `routing_api.RoutingAPI`, so
that an application which already has its own route model, its own ranking layer and its
own map can adopt detour reranking without adopting the demo architecture.

`RoutingAPI` is to the [HERE Routing API v8][1] what `api.API` is to the
[HERE Geocoding & Search API][2]: same constructor arguments, same `send()`/`do_send()`
split, same `Request`/`Response` objects, same cache. `RouteEngine` and `DetourRanker`
themselves are only users of it.

[1]: https://www.here.com/docs/bundle/routing-api-v8-api-reference/page/index.html
[2]: https://docs.here.com/geocoding-and-search/reference

In [ ]:
from here_search_demo.auth import Credentials
from here_search_demo.http import HTTPSession
from here_search_demo.routing_api import RoutingAPI

credentials = Credentials()
sent = []  # to count the Routing API requests actually sent

routing_api = RoutingAPI(credentials=credentials, on_request_sent=lambda request: sent.append(request))
session = HTTPSession()

start_pos = 48.57436, 7.71556  # Strasbourg, where the trip started
at_pos = 48.03529, 7.37121  # near Colmar, where the driver is now
stop_pos = 45.75049, 4.84418  # Lyon, the destination

response = await routing_api.routes(
    session=session,
    origin=start_pos,
    destination=stop_pos,
    return_fields=("polyline", "summary"),
    spans=("duration", "length"),
)
len(sent), type(response).__name__

`routes()` builds a `Request` and hands it to `send()`. The request is a regular demo
`Request` object, so it prints as a URL and is the cache key:

In [ ]:
print(response.req.full)
response.req.params

### Sections

A routing response is made of routes, each made of *sections*. A route without via
waypoint has one section; **each via waypoint adds one section**, which is the property
detour reranking is built on. `RoutingAPI.sections()` maps them to `RouteSection`
dataclasses, so the caller never walks the JSON:

In [ ]:
sections = RoutingAPI.sections(response)
(section,) = sections

print("duration:", section.duration, "s")
print("length:  ", section.length, "m")
print("polyline:", section.polyline[:40], "...")
print("spans:   ", len(section.spans))

### Waypoints

Origin, destination and via accept either a `(latitude, longitude)` tuple or a `Waypoint`.
A `Waypoint` carries an optional `name_hint`, sent as the Routing API `nameHint` waypoint
option: it helps the router pick the right side of a complex intersection, which matters a
lot when the via waypoint sits on a dual carriageway or a motorway service area.

In [ ]:
from here_search_demo.routing_api import Waypoint

print(str(Waypoint(47.23786, 6.02409)))
print(str(Waypoint(47.23786, 6.02409, name_hint="Avenue Carnot")))  # a via in Besançon

Several via waypoints can be passed at once; they are encoded as repeated `via` parameters:

In [ ]:
request = routing_api._build_routes_request(
    start_pos,
    stop_pos,
    via=[(47.23786, 6.02409, "Avenue Carnot"), Waypoint(46.20501, 5.22565)],  # Besançon, then Bourg-en-Bresse
)
request.params["via"]

### The cache

Like `API`, `RoutingAPI` keeps a `dict` of responses keyed by `Request.key`. Repeating a
request returns the cached `Response` without any network call — the `on_request_sent`
counter does not move. The end-user can therefore toggle the ranking mode
back and forth at no cost:

In [ ]:
calls_before = len(sent)
again = await routing_api.routes(
    session=session,
    origin=start_pos,
    destination=stop_pos,
    return_fields=("polyline", "summary"),
    spans=("duration", "length"),
)
len(sent) - calls_before, again.data is response.data, len(routing_api.cache)

Pass your own mapping to the `cache=` constructor argument to plug a shared or persistent
cache in. Similarly, `log_fn=` receives one line per request, and `options=` sets the
defaults (`transportMode`, `return`, `spans`) of every call:

In [ ]:
from here_search_demo.routing_api import RoutingOptions

RoutingOptions()

### From the route to a search along the route

The geometry comes back as a [flexible polyline](https://github.com/heremaps/flexible-polyline).
Decoding it gives the route waypoints; re-encoding a simplified version gives the polyline
to send to the Search API. Simplification is not cosmetic: the Search API rejects a `route`
parameter holding more than 2000 points, and this route alone holds more than twice that.
`RouteEngine` applies the same ceiling through its `max_waypoints_count` class attribute.
The [route simplification notebook](obm_10_route_simplify.ipynb) opens that black box.

In [ ]:
from flexpolyline import decode, encode
from here_search_demo.widgets.route_geometry import simplify_polyline

waypoints = decode(section.polyline)
search_polyline = encode(simplify_polyline(waypoints, 2000))

len(waypoints), len(decode(search_polyline)), len(search_polyline)

`API.browse()`, `API.discover()` and `API.autosuggest()` accept that polyline together with
a corridor `width` in meters. The request becomes a POST carrying a
`route=<polyline>;w=<width>` body — so it is never limited by a URL length — and the Search
API only returns the places lying inside the corridor:

In [ ]:
from here_search_demo.api import API

api = API(credentials=credentials)

resp = await api.browse(
    session=session,
    latitude=at_pos[0],
    longitude=at_pos[1],
    categories=["100-1000"],  # restaurants
    polyline=search_polyline,
    width=500,
    limit=20,
    lang="en",
)
resp.titles

Being inside the corridor says nothing about the time it takes to drive to those places:
the road network, the one-way streets and the traffic are ignored. The
[detour reranking notebook](obm_12_detour_api.ipynb) turns the list into a *driving time*
order, with one Routing API call per result.

Two side topics come first, both built on the response obtained above and both independent
from each other: [simplifying the route polyline](obm_10_route_simplify.ipynb) sent as a
search corridor, and [placing the search anchor a few minutes ahead of the
driver](obm_11_position_ahead.ipynb).

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>